# Case study: Qwen3-8B-Base on HH-RLHF

This notebook runs the analysis from `01_quickstart.ipynb` on a real model. **Qwen3-8B-Base** acts as a judge: it's shown a dialogue and two candidate replies, A and B, and asked which one is better. Its confidence $f(x)$ is the probability it gives to **A**, and $y = 1$ if human annotators preferred A. The input $x$ is the model's last-token hidden state for the prompt.

**Requirements.** Step 1 runs the 8B model over all of HH-RLHF, so it needs a GPU and the `llm` extras (`pip install -e ".[llm]"`). Every later step works from the saved outputs and also runs on a CPU.

To analyze a different pair from the paper, change `DATASET` and `MODEL` below. The options are `hhrlhf_judge`, `mmlu`, `mmlu_pro` and `medmcqa`, with any Hugging Face causal LM. Steps 1 to 3 and 5 are dataset-agnostic; step 4 uses HH-RLHF's metadata.

In [ ]:
import json, subprocess, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap
from sklearn.isotonic import IsotonicRegression

from calib_atlas.data.dataset import DATASETS_DIR
from calib_atlas.field import estimate_field, load_run, make_run_loaders
from calib_atlas.models.baselines import TemperatureScaler
from calib_atlas.utils.eval_utils import (
    apply_range_aware_correction,
    fit_range_aware_scale,
    get_smece,
    plot_binned_relplot,
)

REPO = Path.cwd().resolve().parent  # notebooks/ lives in the repository root
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
FIELD_CMAP = LinearSegmentedColormap.from_list("field", ["steelblue", "lightgray", "salmon"])
REGION_COLORS = {"over": "steelblue", "good": "gray", "under": "salmon"}


def run(*args):
    """Run a command from the repository root and print the last lines of its log (minus progress bars)."""
    res = subprocess.run([sys.executable, *args], cwd=REPO, capture_output=True, text=True)
    lines = [l for l in (res.stdout + res.stderr).splitlines() if l.strip() and "it/s" not in l and "%|" not in l]
    print("\n".join(lines[-8:]).replace(str(REPO), "."))
    res.check_returncode()

In [ ]:
DATASET = "hhrlhf_judge"
MODEL = "Qwen/Qwen3-8B-Base"
MODEL_NAME = MODEL.split("/")[-1]
DATA_DIR = Path(DATASETS_DIR) / DATASET / MODEL_NAME

## 1. Extract confidences and embeddings

`generate_real_data` prompts the model on every example and writes:
* `probas.jsonl`: the confidence for each example,
* `embeddings/`: a Hugging Face dataset with one hidden-state vector per example,
* `embeddings/config.json`: records the embedding dimension.

This step is skipped if the files already exist. You can also run the same command from a terminal.

In [ ]:
if not (DATA_DIR / "embeddings").exists():
    run("-m", "calib_atlas.data.generate_real_data", "--dataset", DATASET,
        "--model_name", MODEL, "--output_root", DATASETS_DIR, "--device", "auto")
in_dim = json.loads((DATA_DIR / "embeddings" / "config.json").read_text())["embed_dim"]
print("embedding dimension:", in_dim)

## 2. Train $\phi$

The paper chooses the bandwidth `sigma` and mass regularizer `lambda_mass` separately for each model–dataset pair. It sweeps a grid with `bin/run_models.sh` and keeps the run with the best validation proxy (`bin/find_best_run.py`; see the README). Here we train a single setting. On a GPU this takes a few minutes; on a CPU it's much slower.

In [ ]:
SIGMA, LAMBDA = 0.5, 1e-4
run_dir = REPO / "outputs" / DATASET / MODEL_NAME / "case_study"
run(
    "-m", "calib_atlas.train", "--config-name=config",
    f"++data.name={DATASET}", f"++data.model_name={MODEL_NAME}", f"++model.in_dim={in_dim}",
    f"++loss.sigma={SIGMA}", f"++loss.lambda_mass={LAMBDA}",
    f"++device={DEVICE}", "run_name=case_study",
)

## 3. Estimate the field and find the hidden regions

In [ ]:
phi, cfg = load_run(run_dir, device=DEVICE)
loaders = make_run_loaders(cfg)
test = estimate_field(phi, cfg, bank_loader=loaders["train"], query_loader=loaders["test"])
val = estimate_field(phi, cfg, bank_loader=loaders["train"], query_loader=loaders["val"])
test = {k: v.cpu() if torch.is_tensor(v) else v for k, v in test.items()}
val = {k: v.cpu() if torch.is_tensor(v) else v for k, v in val.items()}

f_t, y_t, h_t = test["f"].numpy(), test["y"].numpy(), test["hat_delta"].numpy()
print(f"test examples: {len(f_t)};  field: mean {h_t.mean():+.3f}, std {h_t.std():.3f}")

The spread of $\hat\delta_\phi$ shows how much calibration varies across inputs beyond what confidence alone explains. A field concentrated near 0 means little hidden structure; a wide one means large regional errors.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.6))
ax.hist(h_t, bins=60, color="gray")
for s in (-0.05, 0.05):
    ax.axvline(s, color="k", lw=0.8, ls="--")
ax.set(xlabel=r"$\hat\delta_\phi(x)$", ylabel="test examples", title="Estimated field")
plt.tight_layout(); plt.show()

In [ ]:
def regions(hat_delta, t):
    """over: field < -t (accuracy below confidence); under: field > t; good: in between."""
    h = np.asarray(hat_delta)
    return np.where(h < -t, "over", np.where(h > t, "under", "good"))

In [ ]:
def region_relplots(f, y, region, n_bins=10):
    names = ["all", "over", "good", "under"]
    fig, axs = plt.subplots(1, 4, figsize=(11, 2.8), sharey=True)
    for ax, name in zip(axs, names):
        m = np.ones_like(region, dtype=bool) if name == "all" else region == name
        if m.sum() == 0:
            ax.set_visible(False)
            continue
        cmap = None if name == "all" else LinearSegmentedColormap.from_list(name, ["lightgray", REGION_COLORS[name]])
        plot_binned_relplot(f[m], y[m], n_bins=n_bins, cmap=cmap, fig=fig, ax=ax)
        ax.set_title(f"{name}  (n={m.sum()})", fontsize=10)
        if ax is not axs[0]:
            ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
T = 0.05
test_region = regions(h_t, T)
region_relplots(f_t, y_t, test_region)

## 4. What is in each region?

The field is learned without any metadata, so we can use HH-RLHF's own labels afterwards to describe what it found. Each example comes from either the *helpful* or the *harmless* subset. Below are each region's subset composition and the dialogues with the most extreme field values.

In [ ]:
meta = pd.read_csv(Path(DATASETS_DIR) / DATASET / "processed_data.csv", usecols=["id", "subset", "prompt"])
df = pd.DataFrame({"id": [int(i) for i in test["id"]], "f": f_t, "y": y_t, "hat_delta": h_t, "region": test_region})
df = df.merge(meta, on="id", how="left")
pd.crosstab(df["region"], df["subset"], normalize="index").round(3).loc[["over", "good", "under"]]

In [ ]:
pd.set_option("display.max_colwidth", 160)
for name, asc in [("most overconfident", True), ("most underconfident", False)]:
    print(f"--- {name}")
    display(df.sort_values("hat_delta", ascending=asc).head(3)[["hat_delta", "f", "y", "subset", "prompt"]])

## 5. Local confidence correction

As in the quickstart, $\alpha$ is fitted on the validation split and every method is scored on the test split. Confidence-only methods (temperature scaling, isotonic regression) can fix the aggregate curve. The field-based correction can also fix the regions where the model is over- or underconfident.

In [ ]:
def correct(val, test):
    """Fit every recalibration map on the validation split, apply it to the test split."""
    fv, yv, hv = val["f"].numpy(), val["y"].numpy(), val["hat_delta"].numpy()
    ft = test["f"].numpy()
    alpha = fit_range_aware_scale(fv, yv, hv)  # strength of the local, field-based correction
    ts = TemperatureScaler().fit(fv, yv)
    iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(fv, yv)
    return {
        "uncorrected": ft,
        "temperature": np.asarray(ts.predict_proba(ft)),
        "isotonic": iso.predict(ft),
        "field (ours)": apply_range_aware_correction(test["f"], test["hat_delta"], alpha).numpy(),
    }

In [ ]:
def compare_by_region(y, region, preds):
    """smECE and Brier score of each prediction column, overall and per region."""
    rows = []
    for name in ["all", "over", "good", "under"]:
        m = np.ones_like(region, dtype=bool) if name == "all" else region == name
        if m.sum() == 0:
            continue
        for method, p in preds.items():
            rows.append({"region": name, "n": int(m.sum()), "method": method,
                         "smECE": get_smece(p[m], y[m]), "Brier": float(np.mean((p[m] - y[m]) ** 2))})
    df = pd.DataFrame(rows)
    return df.pivot_table(index=["region", "n"], columns="method", values=["smECE", "Brier"], sort=False).round(4)

In [ ]:
preds = correct(val, test)
compare_by_region(y_t, test_region, preds)